# Importing Libraries

In [1]:
import requests
import json
import pandas as pd
import openpyxl
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
import csv
import re

In [31]:

master_url = "https://eraktkosh.mohfw.gov.in/eraktkoshPortal/eraktkosh/master/all"
stock_url = "https://eraktkosh.mohfw.gov.in/eraktkoshPortal/eraktkosh/blood-availability" 

# Initialization of Main Function

In [32]:
def fetch_master_all():
    headers = {
    'Content-Type' : 'application/json'
    }
    response = requests.post(master_url,json={"hospitalCode": 100}, headers=headers)
    response.raise_for_status()
    payload = response.json()
    state_dict = {}
    district_dict = {}
    for state in payload.get("statesWithDistricts", []):
        state_code = state["stateCode"]
        state_dict[state["stateName"]] = state_code
        district_dict[state_code] = {
            d["districtName"]: d["districtCode"] for d in state.get("districts", [])
        }
    blood_dict = {g["bloodGroupName"]: g["bloodGroupCode"] for g in payload.get("bloodGroups", [])}
    component_dict = {c["componentName"]: c["componentCode"] for c in payload.get("componentList", [])}
    return state_dict, district_dict, blood_dict, component_dict

In [33]:
def save_master_data(path="master_data.json"):
    """Cache master data to disk so you don't refetch it on every collection run."""
    state_dict, district_dict, blood_dict, component_dict = fetch_master_all()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "state_dict": state_dict,
                "district_dict": district_dict,
                "blood_dict": blood_dict,
                "component_dict": component_dict,
            },
            f, ensure_ascii=False, indent=2,
        )
    return state_dict, district_dict, blood_dict, component_dict

In [34]:
def load_master_data(path="master_data.json"):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["state_dict"], data["district_dict"], data["blood_dict"], data["component_dict"]

In [35]:
def fetch_blood_data(state_code, district_code, blood_group_code, component_code, component_name):
    params = {
        "stateCode": state_code,
        "districtId": district_code,
        "componentId": component_code,
        "bloodGroupId": blood_group_code,
    }
    response = requests.get(stock_url, params=params)
    response.raise_for_status()
    entries = response.json()
    fetched_at = datetime.now().isoformat(timespec="seconds")
    cleaned = []
    for entry in entries:
        comp_info = entry.get("components", {}).get(component_name, {})
        cleaned.append({
            "fetched_at": fetched_at,
            "state_code": state_code,
            "district_code": district_code,
            "blood_group": blood_group_code,
            "blood_component": component_code,
            "blood_bank": entry.get("hospitalname"),
            "hospital_code": entry.get("hospitalCode"),
            "address": entry.get("hospitaladd"),
            "contact": entry.get("hospitalcontact"),
            "category": entry.get("hospitalType"),
            "available": comp_info.get("available_WithQty", ""),
            "not_available": comp_info.get("not_available_WithQty", ""),
            "last_updated": entry.get("entrydate"),
            "bank_type": entry.get("type"),
        })
    return cleaned

In [37]:
rows = fetch_blood_data("28", "545", "13", "11", "Whole Blood")
print(rows)

[{'fetched_at': '2026-08-23T20:43:12', 'state_code': '28', 'district_code': '545', 'blood_group': '13', 'blood_component': '11', 'blood_bank': 'Dhanwantani Voluntary Blood Bank', 'hospital_code': '28106', 'address': 'Rajahmundry, East Godavari, Andhra Pradesh', 'contact': 'Phone: - ,Fax: -, Email: rjydvbb@gmail.com', 'category': 'Charitable/Vol', 'available': 'B+Ve : 5', 'not_available': '', 'last_updated': '2026-08-23 10:19:08', 'bank_type': 'Blood Bank'}, {'fetched_at': '2026-08-23T20:43:12', 'state_code': '28', 'district_code': '545', 'blood_group': '13', 'blood_component': '11', 'blood_bank': 'Bsu Chc Kovvuru', 'hospital_code': '281755', 'address': 'Kovvuru, , KOVVUR, East Godavari, Andhra Pradesh', 'contact': '-', 'category': 'Govt.', 'available': '', 'not_available': 'B+Ve : 0', 'last_updated': '2026-08-23 08:08:30', 'bank_type': 'BSU'}, {'fetched_at': '2026-08-23T20:43:12', 'state_code': '28', 'district_code': '545', 'blood_group': '13', 'blood_component': '11', 'blood_bank': 'D